# EduHinglish: Gemma-3-4B-it Fine-Tuning (T4 GPU)
This notebook fine-tunes Google's Gemma-3-4B-it model on the combined Biology and Social Science Hinglish dataset using QLoRA.

> **Runtime**: Runtime → Change runtime type → T4 GPU

In [ ]:
!pip install -q -U bitsandbytes transformers peft accelerate datasets trl
!pip install -q huggingface_hub

## 1. Authentication & Drive Mount
Gemma is a gated model, so you **must** have a HuggingFace account (it's completely free).
1. Go to [HuggingFace](https://huggingface.co/google/gemma-3-4b-it) and click **Acknowledge license**.
2. Go to [Settings → Tokens](https://huggingface.co/settings/tokens) and create a **Read** token.
3. In Colab, click the **🔑 Secrets** icon on the left sidebar → add a new secret named `HF_TOKEN` → paste your token.
4. We will also mount Google Drive since your datasets are stored there.

In [ ]:
from google.colab import drive, userdata
from huggingface_hub import login

# Mount Google Drive
drive.mount('/content/drive')

# Login to HuggingFace using the Colab Secret
hf_token = userdata.get('HF_TOKEN')
login(hf_token)

## 2. Load and Prepare Dataset
Loading `bio_9_10.json` and `ss9.json` directly from your Google Drive folder.

In [ ]:
import json
import random
from datasets import Dataset

base_dir = "/content/drive/MyDrive/EduHinglish-ollama"

# Load datasets from Drive
try:
    with open(f"{base_dir}/data/bio_9_10.json", "r") as f:
        bio_data = json.load(f)
    with open(f"{base_dir}/data/ss9.json", "r") as f:
        ss9_data = json.load(f)
except FileNotFoundError:
    print(f"Could not find datasets in {base_dir}/data/ — please check the path.")
    bio_data, ss9_data = [], []

all_data = bio_data + ss9_data
random.seed(42)
random.shuffle(all_data)

print(f"Total entries loaded: {len(all_data)}")

### Format for Gemma Chat Template
We format each entry into Gemma's native conversational format.

In [ ]:
def format_gemma_chat(entry):
    context = entry.get("ncert_context", "")
    question = entry.get("question_english", "")
    intent = entry.get("intent", "")

    prompt = (
        f"Context:\n{context}\n\n"
        f"Question:\n{question}\n\n"
        f"Intent Type: {intent}\n"
        f"Please provide the explanation in Hinglish."
    )
    # Prefer Hinglish answer; fall back to English
    answer = entry.get("answer_hinglish") or entry.get("answer_english", "")

    # Gemma chat template
    formatted_text = (
        f"<start_of_turn>user\n{prompt}<end_of_turn>\n"
        f"<start_of_turn>model\n{answer}<end_of_turn>"
    )
    return {"text": formatted_text}

formatted_data = [format_gemma_chat(e) for e in all_data]
dataset = Dataset.from_list(formatted_data)
print(f"Formatted {len(dataset)} entries.")

### Train / Test Split (95 / 5)

In [ ]:
split_dataset = dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = split_dataset["train"]
eval_dataset  = split_dataset["test"]

print(f"Training samples  : {len(train_dataset)}")
print(f"Evaluation samples: {len(eval_dataset)}")
print("\nSample entry:")
print(train_dataset[0]["text"])

## 3. Model Configuration (QLoRA for T4)
Loading Gemma-3-4B-it in 4-bit NF4 precision. We use `fp16=True` because the T4 does not support bfloat16.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_id = "google/gemma-3-4b-it"

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# T4 does not support bfloat16 — use float16 here
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

### LoRA Adapters

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 4. Training
⚠️ **T4 / Free Colab tip**: Checkpoints are saved every 100 steps to your Google Drive so you don't lose progress if the session disconnects.

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir=f"{base_dir}/training/gemma-eduhinglish-v1",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,      # effective batch size = 8
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,                     # saves to Drive every 100 steps
    save_total_limit=3,                 # keep only last 3 checkpoints to save Drive space
    max_steps=500,                      # ~1 pass through dataset; increase for more epochs
    optim="paged_adamw_8bit",
    fp16=True,                          # T4 uses fp16, NOT bf16
    bf16=False,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    group_by_length=True,
    lr_scheduler_type="cosine",
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=lora_config,
    dataset_text_field="text",
    max_seq_length=1024,
    tokenizer=tokenizer,
    args=training_args,
)

print("Starting Training...")
trainer.train()

## 5. Save Adapter & Test Inference

In [ ]:
# Save the LoRA adapter to Google Drive
adapter_path = f"{base_dir}/training/gemma-eduhinglish-adapter"
trainer.model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"Adapter saved to: {adapter_path}")

In [ ]:
# Quick inference test on an eval sample
def generate_response(prompt_text, max_new_tokens=200):
    inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

test_entry = eval_dataset[0]["text"]
prompt_only = test_entry.split("<start_of_turn>model")[0] + "<start_of_turn>model\n"

print("--- PROMPT ---")
print(prompt_only)
print("\n--- GENERATED HINGLISH RESPONSE ---")
print(generate_response(prompt_only))